### CLASE CLIENTE

In [ ]:
class Cliente:
    def __init__(self, dni, nombre, apellido, telefono=""):
        self.__dni = dni
        self.__nombre = nombre
        self.__apellido = apellido
        self.__telefono = telefono
        self.__cuentas = []

# ---------------- GETTERS ----------------
    def get_dni(self):
        return self.__dni

    def get_nombre_completo(self):
        return self.__nombre + " " + self.__apellido

    def get_telefono(self):
        return self.__telefono

    def get_cuentas(self):
        return self.__cuentas
    
# ---------------- SETTERS ----------------
    def set_telefono(self, nuevo_telefono):
        # Validacion: un celular en Peru tiene 9 digitos
        if nuevo_telefono.isdigit() and len(nuevo_telefono) == 9:
            self.__telefono = nuevo_telefono
        else:
            print("Telefono no valido: debe tener 9 digitos.")

# ---------------- METODOS ----------------
    def agregar_cuenta(self, cuenta):
        self.__cuentas.append(cuenta)

    def saldo_total(self):
        total = 0
        for cuenta in self.__cuentas:
            total += cuenta.get_saldo()
        return total

    def mostrar(self):
        print(f"\nCliente: {self.get_nombre_completo()}  |  DNI: {self.__dni}"
              f"  |  Tel: {self.__telefono}")
        if len(self.__cuentas) == 0:
            print("   (no tiene cuentas)")
        for cuenta in self.__cuentas:
            print("  ", cuenta)
        print(f"   Saldo total: S/ {self.saldo_total():.2f}")

    def __str__(self):
        return f"{self.__dni} - {self.get_nombre_completo()} ({len(self.__cuentas)} cuenta/s)"

    

### CLASE BANCO

In [4]:
class Banco:
    def __init__(self, nombre):
        self.__nombre = nombre
        self.__clientes = []     # LISTA de objetos Cliente
        self.__cuentas = []      # LISTA de objetos Cuenta

    def get_nombre(self):
        return self.__nombre

    def get_clientes(self):
        return self.__clientes

# ---------------- CLIENTES ----------------
    def buscar_cliente(self, dni):
        for cliente in self.__clientes:
            if cliente.get_dni() == dni:
                return cliente
        return None

    def registrar_cliente(self, dni, nombre, apellido, telefono=""):
        # Validaciones
        if not (dni.isdigit() and len(dni) == 8):
            print("DNI no valido: debe tener 8 digitos.")
            return None
        if self.buscar_cliente(dni) is not None:
            print("Ya existe un cliente con ese DNI.")
            return None
        cliente = Cliente(dni, nombre, apellido, telefono)
        self.__clientes.append(cliente)
        print(f"Cliente {cliente.get_nombre_completo()} registrado correctamente.")
        return cliente

# ---------------- CUENTAS ----------------
    def abrir_cuenta(self, dni, tipo, saldo_inicial=0):
        """tipo = 1 (Ahorros) o 2 (Corriente)"""
        cliente = self.buscar_cliente(dni)
        if cliente is None:
            print("No existe un cliente con ese DNI.")
            return None
        if tipo == 1:
            cuenta = CuentaAhorros(cliente, saldo_inicial)
        elif tipo == 2:
            cuenta = CuentaCorriente(cliente, saldo_inicial)
        else:
            print("Tipo de cuenta no valido.")
            return None
        cliente.agregar_cuenta(cuenta)
        self.__cuentas.append(cuenta)
        print(f"Se abrio la {cuenta.get_tipo()} N° {cuenta.get_numero()} "
              f"para {cliente.get_nombre_completo()}.")
        return cuenta

    def buscar_cuenta(self, numero):
        for cuenta in self.__cuentas:
            if cuenta.get_numero() == numero:
                return cuenta
        return None
    
# ---------------- OPERACIONES ----------------
    def transferir(self, numero_origen, numero_destino, monto):
        origen = self.buscar_cuenta(numero_origen)
        destino = self.buscar_cuenta(numero_destino)
        if origen is None or destino is None:
            print("Una de las cuentas no existe.")
            return False
        if origen == destino:
            print("No se puede transferir a la misma cuenta.")
            return False
        # Primero se intenta retirar; solo si funciona se deposita en el destino
        if origen.retirar(monto, "Transferencia enviada"):
            destino.depositar(monto, "Transferencia recibida")
            print(f"Transferencia exitosa de S/ {monto:.2f}: "
                  f"{origen.get_numero()} -> {destino.get_numero()}")
            return True
        return False

    def cierre_de_mes(self):
        print("\n=== CIERRE DE MES ===")
        for cuenta in self.__cuentas:
            saldo_antes = cuenta.get_saldo()
            cuenta.cierre_de_mes()          # POLIMORFISMO: cada cuenta hace lo suyo
            diferencia = cuenta.get_saldo() - saldo_antes
            print(f"  {cuenta.get_numero()} {cuenta.get_tipo():<17} "
                  f"cambio: S/ {diferencia:>8.2f}   nuevo saldo: S/ {cuenta.get_saldo():.2f}")
            